# 02 — Analytical Dataset

## Customer Intelligence Suite — Retail Banking

This notebook builds the analytical datasets used by the downstream Customer Intelligence and Credit Risk workflows.

The objective is to transform the normalized relational data stored in PostgreSQL into analysis-ready datasets with explicitly defined observation units and temporal boundaries.

Two analytical datasets will be considered:

1. **Customer Intelligence Dataset**
   - Observation unit: bank account.
   - Purpose: descriptive customer and account-level behavioral analysis.
   - May use the available transaction history to characterize account behavior.

2. **Credit Risk Dataset**
   - Observation unit: loan.
   - Purpose: predictive credit-risk modeling.
   - Features must only use information available before the corresponding loan origination date to prevent data leakage.

The datasets are constructed separately because their analytical objectives and temporal requirements are different.

In [1]:
import sys

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt

In [2]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.database import engine

with engine.connect():
    print('PostgreSQL connection successful.')

PostgreSQL connection successful.


## 1. Analytical Design

### 1.1 Observation Units

The normalized database contains several one-to-many relationships. Therefore, directly joining all tables would create duplicated observations and distort aggregated metrics.

The analytical datasets are constructed at two different granularities.

#### Customer Intelligence

One observation represents one bank account:

`account_id`

The account owner is identified through `disp.type = 'OWNER'`. Additional dispositions are treated as account-level characteristics rather than independent account observations.

#### Credit Risk

One observation represents one loan:

`loan_id`

The loan origination date defines the temporal cutoff for predictive features. Transactional information occurring after this date must not be used as predictor information.

In [3]:
query = '''
SELECT
    COUNT(*) AS total_accounts,
    COUNT(*) FILTER (WHERE n_owners = 1) AS accounts_with_one_owner,
    COUNT(*) FILTER (WHERE n_owners = 0) AS accounts_without_owner,
    COUNT(*) FILTER (WHERE n_owners > 1) AS accounts_with_multiple_owners
FROM (
    SELECT
        a.account_id,
        COUNT(d.disp_id) FILTER (
            WHERE d.type = 'OWNER'
        ) AS n_owners
    FROM account a
    LEFT JOIN disp d
        ON a.account_id = d.account_id
    GROUP BY a.account_id
) account_owners;
'''

owner_validation = pd.read_sql(
    query,
    engine
)

owner_validation

,total_accounts,accounts_with_one_owner,accounts_without_owner,accounts_with_multiple_owners
0,4500,4500,0,0


### 1.2 Customer Intelligence Base Population

All 4,500 accounts have exactly one `OWNER`. Therefore, `account_id` provides a consistent observation unit for the Customer Intelligence analytical dataset.

The base population combines each account with its owner and the owner's demographic information. Account-level product and behavioral features will be added subsequently through aggregated tables to preserve one row per account.

In [4]:
query = '''
SELECT
    a.account_id,
    a.date AS account_open_date,
    a.frequency,

    c.client_id AS owner_client_id,
    c.birth_date,
    c.gender,
    c.age,

    a.district_id AS account_district_id,
    c.district_id AS client_district_id

FROM account a

INNER JOIN disp d
    ON a.account_id = d.account_id
    AND d.type = 'OWNER'

INNER JOIN client c
    ON d.client_id = c.client_id

ORDER BY a.account_id;
'''

customer_base = pd.read_sql(
    query,
    engine
)

customer_base.head()

,account_id,account_open_date,frequency,owner_client_id,birth_date,gender,age,account_district_id,client_district_id
0,1,1995-03-24,MONTHLY,1,1970-12-13,Female,28,18,18
1,2,1993-02-26,MONTHLY,2,1945-02-04,Male,53,1,1
2,3,1997-07-07,MONTHLY,4,1956-12-01,Male,42,5,5
3,4,1996-02-21,MONTHLY,6,1919-09-22,Male,79,12,12
4,5,1997-05-30,MONTHLY,7,1929-01-25,Male,69,15,15


In [5]:
customer_base.shape

(4500, 9)

In [6]:
customer_base['account_id'].nunique()

4500

In [7]:
customer_base.isna().sum()

account_id             0
account_open_date      0
frequency              0
owner_client_id        0
birth_date             0
gender                 0
age                    0
account_district_id    0
client_district_id     0
dtype: int64

In [8]:
customer_base['owner_client_id'].nunique()

4500

### 1.3 District Characteristics

The account and owner district identifiers are compared before incorporating socioeconomic information. The identifiers differ for 409 of the 4,500 accounts (9.1%), indicating that account location and customer location should be treated as distinct concepts.

Both district identifiers are therefore retained. Socioeconomic characteristics are joined using the owner's `client_district_id`, as these variables are intended to describe the customer's demographic and socioeconomic context. A `district_mismatch` indicator is also retained to identify accounts whose registered district differs from that of their owner.

In [9]:
district_mismatch = (
    customer_base['account_district_id']
    != customer_base['client_district_id']
).sum()

district_mismatch

np.int64(409)

In [10]:
customer_base['district_mismatch'] = (
    customer_base['account_district_id']
    != customer_base['client_district_id']
)

customer_base['district_mismatch'].value_counts()

district_mismatch
False    4091
True      409
Name: count, dtype: int64

In [11]:
query = '''
SELECT
    district_id AS client_district_id,
    district_name,
    region,
    inhabitants,
    urban_ratio,
    average_salary,
    unemployment_rate_96,
    entrepreneurs_per_1000,
    crimes_96
FROM district
ORDER BY district_id;
'''

district_features = pd.read_sql(
    query,
    engine
)

district_features.head()

,client_district_id,district_name,region,inhabitants,urban_ratio,average_salary,unemployment_rate_96,entrepreneurs_per_1000,crimes_96
0,1,Hl.m. Praha,Prague,1204953,100.0,12541.0,0.43,167.0,99107
1,2,Benesov,central Bohemia,88884,46.7,8507.0,1.85,132.0,2674
2,3,Beroun,central Bohemia,75232,41.7,8980.0,2.21,111.0,2813
3,4,Kladno,central Bohemia,149893,67.4,9753.0,5.05,109.0,5892
4,5,Kolin,central Bohemia,95616,51.4,9307.0,4.43,118.0,3040


In [12]:
customer_base = customer_base.merge(
    district_features,
    on='client_district_id',
    how='left',
    validate='many_to_one'
)

In [13]:
customer_base.head()

,account_id,account_open_date,frequency,owner_client_id,birth_date,gender,age,account_district_id,client_district_id,district_mismatch,district_name,region,inhabitants,urban_ratio,average_salary,unemployment_rate_96,entrepreneurs_per_1000,crimes_96
0,1,1995-03-24,MONTHLY,1,1970-12-13,Female,28,18,18,False,Pisek,south Bohemia,70699,65.3,8968.0,3.35,131.0,1910
1,2,1993-02-26,MONTHLY,2,1945-02-04,Male,53,1,1,False,Hl.m. Praha,Prague,1204953,100.0,12541.0,0.43,167.0,99107
2,3,1997-07-07,MONTHLY,4,1956-12-01,Male,42,5,5,False,Kolin,central Bohemia,95616,51.4,9307.0,4.43,118.0,3040
3,4,1996-02-21,MONTHLY,6,1919-09-22,Male,79,12,12,False,Pribram,central Bohemia,107870,58.0,8754.0,4.31,137.0,3868
4,5,1997-05-30,MONTHLY,7,1929-01-25,Male,69,15,15,False,Cesky Krumlov,south Bohemia,58796,51.9,9045.0,3.60,124.0,1879


In [14]:
customer_base.shape

(4500, 18)

In [15]:
customer_base.isna().sum()

account_id                0
account_open_date         0
frequency                 0
owner_client_id           0
birth_date                0
gender                    0
age                       0
account_district_id       0
client_district_id        0
district_mismatch         0
district_name             0
region                    0
inhabitants               0
urban_ratio               0
average_salary            0
unemployment_rate_96      0
entrepreneurs_per_1000    0
crimes_96                 0
dtype: int64

### 1.4 Account Relationships and Products

Account-level product information is aggregated before being merged with the analytical population. This preserves the one-row-per-account structure while incorporating information about authorized users, payment cards, and other banking relationships.

Each source table is aggregated independently before joining to prevent row multiplication caused by one-to-many relationships.

#### 1.4.1 Dispositions

In [16]:
query = '''
SELECT
    account_id,
    COUNT(*) AS n_dispositions,
    COUNT(*) FILTER (
        WHERE type = 'DISPONENT'
    ) AS n_disponents
FROM disp
GROUP BY account_id
ORDER BY account_id;
'''

disposition_features = pd.read_sql(
    query,
    engine
)

disposition_features.head()

,account_id,n_dispositions,n_disponents
0,1,1,0
1,2,2,1
2,3,2,1
3,4,1,0
4,5,1,0


In [17]:
disposition_features['has_disponent'] = (
    disposition_features['n_disponents'] > 0
)

In [18]:
disposition_features['n_disponents'].value_counts().sort_index()

n_disponents
0    3631
1     869
Name: count, dtype: int64

In [19]:
disposition_features.head()

,account_id,n_dispositions,n_disponents,has_disponent
0,1,1,0,False
1,2,2,1,True
2,3,2,1,True
3,4,1,0,False
4,5,1,0,False


In [20]:
customer_base = customer_base.merge(
    disposition_features[
        ['account_id', 'has_disponent']
    ],
    on='account_id',
    how='left',
    validate='one_to_one'
)

In [21]:
customer_base['has_disponent'].value_counts()
customer_base.head()

,account_id,account_open_date,frequency,owner_client_id,birth_date,gender,age,account_district_id,client_district_id,district_mismatch,district_name,region,inhabitants,urban_ratio,average_salary,unemployment_rate_96,entrepreneurs_per_1000,crimes_96,has_disponent
0,1,1995-03-24,MONTHLY,1,1970-12-13,Female,28,18,18,False,Pisek,south Bohemia,70699,65.3,8968.0,3.35,131.0,1910,False
1,2,1993-02-26,MONTHLY,2,1945-02-04,Male,53,1,1,False,Hl.m. Praha,Prague,1204953,100.0,12541.0,0.43,167.0,99107,True
2,3,1997-07-07,MONTHLY,4,1956-12-01,Male,42,5,5,False,Kolin,central Bohemia,95616,51.4,9307.0,4.43,118.0,3040,True
3,4,1996-02-21,MONTHLY,6,1919-09-22,Male,79,12,12,False,Pribram,central Bohemia,107870,58.0,8754.0,4.31,137.0,3868,False
4,5,1997-05-30,MONTHLY,7,1929-01-25,Male,69,15,15,False,Cesky Krumlov,south Bohemia,58796,51.9,9045.0,3.60,124.0,1879,False


In [22]:
assert len(customer_base) == 4500
assert customer_base['account_id'].nunique() == 4500

#### 1.4.2 Payment Cards

Cards are linked to dispositions rather than directly to accounts. Therefore, card information is mapped to `account_id` through the `disp` table and aggregated before being incorporated into the analytical dataset.

The resulting features describe whether the account has an associated card and, when applicable, its card type, while preserving one row per account.

In [23]:
query = '''
SELECT
    d.account_id,
    c.card_id,
    c.type AS card_type,
    d.type AS disposition_type
FROM card c
INNER JOIN disp d
    ON c.disp_id = d.disp_id
ORDER BY d.account_id, c.card_id;
'''

account_cards = pd.read_sql(
    query,
    engine
)

account_cards.head()

,account_id,card_id,card_type,disposition_type
0,7,1,gold,OWNER
1,14,2,classic,OWNER
2,33,3,gold,OWNER
3,34,4,classic,OWNER
4,43,5,junior,OWNER


In [24]:
account_cards.groupby('account_id').size().value_counts().sort_index()

1    892
Name: count, dtype: int64

In [25]:
pd.crosstab(
    account_cards['card_type'],
    account_cards['disposition_type']
)

disposition_type,OWNER
card_type,
classic,659
gold,88
junior,145


In [26]:
card_features = account_cards[
    ['account_id', 'card_type']
].copy()

card_features['has_card'] = True

card_features.head()

,account_id,card_type,has_card
0,7,gold,True
1,14,classic,True
2,33,gold,True
3,34,classic,True
4,43,junior,True


In [27]:
customer_base = customer_base.merge(
    card_features,
    on='account_id',
    how='left',
    validate='one_to_one'
)

customer_base['has_card'] = (
    customer_base['has_card']
    .fillna(False)
    .astype(bool)
)

In [28]:
customer_base[
    ['has_card', 'card_type']
].value_counts(
    dropna=False
)

has_card  card_type
False     NaN          3608
True      classic       659
          junior        145
          gold           88
Name: count, dtype: int64

In [29]:
assert len(customer_base) == 4500
assert customer_base['account_id'].nunique() == 4500
assert (
    customer_base.loc[
        ~customer_base['has_card'],
        'card_type'
    ].isna().all()
)

## 1.5 Standing Orders

Standing orders represent recurring payment commitments associated with bank accounts.

Because an account may have multiple standing orders, the source table is aggregated at `account_id` level before being incorporated into the analytical dataset. The objective is to capture the number, monetary magnitude, and purpose of recurring payment commitments while preserving one row per account.

In [30]:
query = '''
SELECT
    account_id,
    COUNT(*) AS n_standing_orders
FROM standing_order
GROUP BY account_id
ORDER BY account_id;
'''

standing_order_counts = pd.read_sql(
    query,
    engine
)

standing_order_counts[
    'n_standing_orders'
].value_counts().sort_index()

n_standing_orders
1    2103
2     949
3     416
4     228
5      62
Name: count, dtype: int64

In [31]:
len(customer_base) - standing_order_counts['account_id'].nunique()

742

In [32]:
query = '''
SELECT
    account_id,
    k_symbol,
    COUNT(*) AS n_orders,
    SUM(amount) AS total_amount
FROM standing_order
GROUP BY
    account_id,
    k_symbol
ORDER BY
    account_id,
    k_symbol;
'''

standing_order_types = pd.read_sql(
    query,
    engine
)

standing_order_types.head(10)

,account_id,k_symbol,n_orders,total_amount
0,1,HOUSEHOLD_PAYMENT,1,2452.0
1,2,HOUSEHOLD_PAYMENT,1,7266.0
2,2,LOAN_PAYMENT,1,3372.7
3,3,HOUSEHOLD_PAYMENT,1,1135.0
4,3,INSURANCE_PAYMENT,1,3539.0
5,3,NaN,1,327.0
6,4,HOUSEHOLD_PAYMENT,2,3363.0
7,5,HOUSEHOLD_PAYMENT,1,2668.0
8,6,HOUSEHOLD_PAYMENT,1,3954.0
9,7,HOUSEHOLD_PAYMENT,1,4880.0


In [33]:
standing_order_types[
    'n_orders'
].value_counts().sort_index()

n_orders
1    5835
2     318
Name: count, dtype: int64

#### Standing Order Features

Standing-order information is summarized at account level. Since accounts may hold multiple orders, both the number of recurring commitments and their monetary magnitude are retained.

Purpose-specific amounts are aggregated separately. A zero value indicates that the account has no standing order in that category. Orders without a recorded `k_symbol` are retained as uncategorized rather than assigned to an inferred purpose.

In [34]:
query = '''
SELECT
    account_id,
    COUNT(*) AS n_standing_orders,
    SUM(amount) AS total_standing_order_amount,

    SUM(amount) FILTER (
        WHERE k_symbol = 'HOUSEHOLD_PAYMENT'
    ) AS household_payment_amount,

    SUM(amount) FILTER (
        WHERE k_symbol = 'INSURANCE_PAYMENT'
    ) AS insurance_payment_amount,

    SUM(amount) FILTER (
        WHERE k_symbol = 'LEASING_PAYMENT'
    ) AS leasing_payment_amount,

    SUM(amount) FILTER (
        WHERE k_symbol = 'LOAN_PAYMENT'
    ) AS loan_payment_amount,

    SUM(amount) FILTER (
        WHERE k_symbol IS NULL
    ) AS uncategorized_order_amount

FROM standing_order
GROUP BY account_id
ORDER BY account_id;
'''

standing_order_features = pd.read_sql(
    query,
    engine
)

standing_order_features.head()

,account_id,n_standing_orders,total_standing_order_amount,household_payment_amount,insurance_payment_amount,leasing_payment_amount,loan_payment_amount,uncategorized_order_amount
0,1,1,2452.0,2452.0,NaN,NaN,NaN,NaN
1,2,2,10638.7,7266.0,NaN,NaN,3372.7,NaN
2,3,3,5001.0,1135.0,3539.0,NaN,NaN,327.0
3,4,2,3363.0,3363.0,NaN,NaN,NaN,NaN
4,5,1,2668.0,2668.0,NaN,NaN,NaN,NaN


In [35]:
standing_order_features['has_standing_order'] = True

In [36]:
customer_base = customer_base.merge(
    standing_order_features,
    on='account_id',
    how='left',
    validate='one_to_one'
)

In [37]:
standing_order_columns = [
    'n_standing_orders',
    'total_standing_order_amount',
    'household_payment_amount',
    'insurance_payment_amount',
    'leasing_payment_amount',
    'loan_payment_amount',
    'uncategorized_order_amount'
]

customer_base[standing_order_columns] = (
    customer_base[standing_order_columns]
    .fillna(0)
)

customer_base['n_standing_orders'] = (
    customer_base['n_standing_orders']
    .astype(int)
)

In [38]:
assert len(customer_base) == 4500
assert customer_base['account_id'].nunique() == 4500

customer_base[
    [
        'has_standing_order',
        'n_standing_orders',
        'total_standing_order_amount'
    ]
].head(10)

,has_standing_order,n_standing_orders,total_standing_order_amount
0,True,1,2452.0
1,True,2,10638.7
2,True,3,5001.0
3,True,2,3363.0
4,True,1,2668.0
5,True,1,3954.0
6,True,1,4880.0
7,True,2,9324.0
8,NaN,0,0.0
9,True,2,8377.0


## 1.6 Loans

Loans represent an additional financial product associated with bank accounts.

For the Customer Intelligence dataset, loan information is used to describe the customer's product relationship and credit exposure. Credit-risk modeling will be handled separately using a loan-level analytical dataset with appropriate temporal controls.

Before incorporating loan features, the relationship between accounts and loans is verified to preserve the one-row-per-account structure.

In [39]:
query = '''
SELECT
    account_id,
    COUNT(*) AS n_loans
FROM loan
GROUP BY account_id
ORDER BY account_id;
'''

loan_counts = pd.read_sql(
    query,
    engine
)

loan_counts['n_loans'].value_counts().sort_index()

n_loans
1    682
Name: count, dtype: int64

In [40]:
len(customer_base) - loan_counts['account_id'].nunique()

3818

In [41]:
query = '''
SELECT
    loan_id,
    account_id,
    date AS loan_date,
    amount AS loan_amount,
    duration AS loan_duration,
    payments AS loan_payment,
    status AS loan_status
FROM loan
ORDER BY account_id;
'''

account_loans = pd.read_sql(
    query,
    engine
)

account_loans.head()

,loan_id,account_id,loan_date,loan_amount,loan_duration,loan_payment,loan_status
0,4959,2,1994-01-05,80952.0,24,3373.0,A
1,4961,19,1996-04-29,30276.0,12,2523.0,B
2,4962,25,1997-12-08,30276.0,12,2523.0,A
3,4967,37,1998-10-14,318480.0,60,5308.0,D
4,4968,38,1998-04-19,110736.0,48,2307.0,C


In [42]:
account_loans[
    [
        'loan_amount',
        'loan_duration',
        'loan_payment'
    ]
].describe()

,loan_amount,loan_duration,loan_payment
count,682.000000,682.000000,682.000000
mean,151410.175953,36.492669,4190.664223
std,113372.406310,17.075219,2215.830344
min,4980.000000,12.000000,304.000000
25%,66732.000000,24.000000,2477.000000
50%,116928.000000,36.000000,3934.000000
75%,210654.000000,48.000000,5813.500000
max,590820.000000,60.000000,9910.000000


In [43]:
account_loans['loan_status'].value_counts().sort_index()

loan_status
A    203
B     31
C    403
D     45
Name: count, dtype: int64

### Loan Features

Since each borrowing account has exactly one loan, a loan count would be redundant with a binary loan indicator.

For Customer Intelligence, loan features describe the presence and magnitude of the customer's credit relationship. Loan performance status is not incorporated into this dataset because it represents a credit outcome and will be handled separately in the Credit Risk analytical dataset.

In [44]:
loan_features = account_loans[
    [
        'account_id',
        'loan_amount',
        'loan_duration',
        'loan_payment'
    ]
].copy()

loan_features['has_loan'] = True

In [45]:
customer_base = customer_base.merge(
    loan_features,
    on='account_id',
    how='left',
    validate='one_to_one'
)

customer_base['has_loan'] = (
    customer_base['has_loan']
    .fillna(False)
    .astype(bool)
)

In [46]:
customer_base[
    [
        'has_loan',
        'loan_amount',
        'loan_duration',
        'loan_payment'
    ]
].isna().groupby(
    customer_base['has_loan']
).sum()

,has_loan,loan_amount,loan_duration,loan_payment
has_loan,,,,
False,0,3818,3818,3818
True,0,0,0,0


In [47]:
assert len(customer_base) == 4500
assert customer_base['account_id'].nunique() == 4500

assert (
    customer_base.loc[
        ~customer_base['has_loan'],
        ['loan_amount', 'loan_duration', 'loan_payment']
    ]
    .isna()
    .all()
    .all()
)

## 1.7 Transaction Behavior

Transaction history provides the main behavioral component of the Customer Intelligence analytical dataset.

The transaction table contains multiple observations per account and is therefore aggregated before being incorporated into the account-level dataset. Transaction features are organized into activity, monetary flows, balance behavior, and operation mix.

The objective is to characterize how accounts are used while preserving the analytical granularity of one row per account.

### 1.7.1 Transaction Activity

Transaction activity features describe the volume and temporal coverage of account usage. Because accounts were opened at different dates, both transaction counts and observed history are considered rather than interpreting raw transaction counts alone.

In [48]:
query = '''
SELECT
    account_id,
    COUNT(*) AS n_transactions,
    MIN(date) AS first_transaction_date,
    MAX(date) AS last_transaction_date,
    COUNT(DISTINCT date) AS active_days
FROM bank_transaction
GROUP BY account_id
ORDER BY account_id;
'''

transaction_activity = pd.read_sql(
    query,
    engine,
    parse_dates=[
        'first_transaction_date',
        'last_transaction_date'
    ]
)

transaction_activity.head()

,account_id,n_transactions,first_transaction_date,last_transaction_date,active_days
0,1,239,1995-03-24,1998-12-31,187
1,2,478,1993-02-26,1998-12-31,369
2,3,117,1997-07-07,1998-12-31,100
3,4,186,1996-02-21,1998-12-31,151
4,5,84,1997-05-30,1998-12-31,69


In [49]:
transaction_activity.shape

(4500, 5)

In [50]:
len(customer_base) - transaction_activity['account_id'].nunique()

0

In [51]:
transaction_activity[
    [
        'n_transactions',
        'active_days'
    ]
].describe()

,n_transactions,active_days
count,4500.000000,4500.000000
mean,234.737778,181.059556
std,126.849443,96.837286
min,9.000000,9.000000
25%,133.000000,103.000000
50%,208.000000,161.000000
75%,330.000000,252.000000
max,675.000000,548.000000


In [52]:
transaction_activity[
    [
        'first_transaction_date',
        'last_transaction_date'
    ]
].agg(['min', 'max'])

,first_transaction_date,last_transaction_date
min,1993-01-01,1996-08-25
max,1997-12-29,1998-12-31


In [53]:
transaction_activity['observation_days'] = (
    transaction_activity['last_transaction_date']
    - transaction_activity['first_transaction_date']
).dt.days + 1

transaction_activity['observation_months'] = (
    transaction_activity['observation_days'] / 30.44
)

transaction_activity['transactions_per_month'] = (
    transaction_activity['n_transactions']
    / transaction_activity['observation_months']
)

In [54]:
transaction_activity[
    [
        'observation_days',
        'observation_months',
        'transactions_per_month'
    ]
].describe()

,observation_days,observation_months,transactions_per_month
count,4500.000000,4500.000000,4500.000000
mean,1239.332222,40.713936,5.634040
std,552.189399,18.140256,1.355429
min,331.000000,10.873850,0.536125
25%,787.000000,25.854139,4.840628
50%,1094.000000,35.939553,5.502566
75%,1829.000000,60.085414,6.362943
max,2191.000000,71.977661,11.726047


In [55]:
transaction_activity['active_day_ratio'] = (
    transaction_activity['active_days']
    / transaction_activity['observation_days']
)

In [56]:
transaction_activity[
    [
        'transactions_per_month',
        'active_day_ratio'
    ]
].describe()

,transactions_per_month,active_day_ratio
count,4500.000000,4500.000000
mean,5.634040,0.143399
std,1.355429,0.033706
min,0.536125,0.017613
25%,4.840628,0.122902
50%,5.502566,0.141509
75%,6.362943,0.160894
max,11.726047,0.263699


### 1.7.2 Credit and Debit Flows

Transaction amounts are aggregated into incoming and outgoing monetary flows to characterize account cash-flow behavior.

`CREDIT` transactions are treated as inflows, while both `DEBIT` and `WITHDRAWAL` transactions are treated as outflows. Transaction counts and monetary amounts are retained separately because they capture different dimensions of account activity.

Flow measures are also normalized by the observed transaction period to improve comparability across accounts with different history lengths.

In [57]:
query = '''
SELECT
    account_id,

    COUNT(*) FILTER (
        WHERE type = 'CREDIT'
    ) AS n_inflows,

    COUNT(*) FILTER (
        WHERE type IN ('DEBIT', 'WITHDRAWAL')
    ) AS n_outflows,

    SUM(amount) FILTER (
        WHERE type = 'CREDIT'
    ) AS total_inflow,

    SUM(amount) FILTER (
        WHERE type IN ('DEBIT', 'WITHDRAWAL')
    ) AS total_outflow

FROM bank_transaction
GROUP BY account_id
ORDER BY account_id;
'''

transaction_flows = pd.read_sql(
    query,
    engine
)

transaction_flows.head()

,account_id,n_inflows,n_outflows,total_inflow,total_outflow
0,1,102,137,194320.5,180854.0
1,2,152,326,1597053.5,1554425.8
2,3,46,71,173059.0,121962.8
3,4,65,121,192346.0,158625.4
4,5,35,49,97485.0,69396.4


In [58]:
transaction_flows[
    [
        'n_inflows',
        'n_outflows',
        'total_inflow',
        'total_outflow'
    ]
].describe()

,n_inflows,n_outflows,total_inflow,total_outflow
count,4500.000000,4500.000000,4.500000e+03,4.500000e+03
mean,90.018444,144.719333,7.172161e+05,6.734047e+05
std,44.678440,87.818427,6.702457e+05,6.585890e+05
min,6.000000,1.000000,1.970000e+04,1.400000e+03
25%,54.000000,77.000000,2.280506e+05,1.990796e+05
50%,81.000000,127.000000,4.666833e+05,4.256768e+05
75%,126.000000,199.000000,1.005419e+06,9.469219e+05
max,272.000000,515.000000,3.857257e+06,3.761845e+06


In [59]:
(
    transaction_flows['n_inflows']
    + transaction_flows['n_outflows']
    == transaction_activity['n_transactions']
).value_counts()

True    4500
Name: count, dtype: int64

In [60]:
transaction_flows = transaction_flows.merge(
    transaction_activity[
        ['account_id', 'observation_months']
    ],
    on='account_id',
    how='left',
    validate='one_to_one'
)

In [61]:
transaction_flows['monthly_inflow'] = (
    transaction_flows['total_inflow']
    / transaction_flows['observation_months']
)

transaction_flows['monthly_outflow'] = (
    transaction_flows['total_outflow']
    / transaction_flows['observation_months']
)

transaction_flows['monthly_net_flow'] = (
    transaction_flows['monthly_inflow']
    - transaction_flows['monthly_outflow']
)

In [62]:
transaction_flows['avg_inflow_amount'] = (
    transaction_flows['total_inflow']
    / transaction_flows['n_inflows']
)

transaction_flows['avg_outflow_amount'] = (
    transaction_flows['total_outflow']
    / transaction_flows['n_outflows']
)

In [63]:
transaction_flows[
    [
        'monthly_inflow',
        'monthly_outflow',
        'monthly_net_flow',
        'avg_inflow_amount',
        'avg_outflow_amount'
    ]
].describe()

,monthly_inflow,monthly_outflow,monthly_net_flow,avg_inflow_amount,avg_outflow_amount
count,4500.000000,4500.000000,4500.000000,4500.000000,4500.000000
mean,17587.150460,16286.794745,1300.355715,7769.748074,4639.858406
std,13518.953370,13173.512671,973.631371,5456.384077,3403.192144
min,919.566228,118.049861,-438.089327,851.402857,498.652941
25%,6165.233333,5247.759181,616.064323,2951.981906,1838.044950
50%,14610.321010,13246.959150,1048.422913,6599.054928,3622.046500
75%,24858.033141,23037.645165,1686.042121,11688.857597,6598.083353
max,75513.474548,72635.156016,7358.617970,26885.955385,19742.525000


In [64]:
transaction_flows[
    ['n_inflows', 'n_outflows']
].eq(0).sum()

n_inflows     0
n_outflows    0
dtype: int64

### 1.7.3 Balance Behavior

Account balances provide information about the level and variability of funds maintained over the observed transaction history.

Balance features are calculated from the post-transaction balance recorded for each transaction. The analysis captures typical balance levels, balance variability, and exposure to negative balances while preserving the account-level analytical granularity.

In [65]:
query = '''
SELECT
    account_id,

    AVG(balance) AS avg_balance,
    MIN(balance) AS min_balance,
    MAX(balance) AS max_balance,
    STDDEV_SAMP(balance) AS balance_std,

    COUNT(*) FILTER (
        WHERE balance < 0
    ) AS n_negative_balance_transactions

FROM bank_transaction
GROUP BY account_id
ORDER BY account_id;
'''

balance_features = pd.read_sql(
    query,
    engine
)

balance_features.head()

,account_id,avg_balance,min_balance,max_balance,balance_std,n_negative_balance_transactions
0,1,15994.008368,1000.0,30414.8,3656.890437,0
1,2,36540.704393,1100.0,69302.0,11028.141344,0
2,3,26973.506838,1000.0,53446.5,10508.662466,0
3,4,21753.446237,800.0,34870.0,5009.975052,0
4,5,25090.069048,600.0,32036.2,4752.471004,0


In [66]:
balance_features[
    [
        'avg_balance',
        'min_balance',
        'max_balance',
        'balance_std',
        'n_negative_balance_transactions'
    ]
].describe()

,avg_balance,min_balance,max_balance,balance_std,n_negative_balance_transactions
count,4500.000000,4500.000000,4500.000000,4500.000000,4500.000000
mean,36667.863523,431.130289,73181.948578,12979.934464,0.666444
std,15165.626204,2359.776113,39590.552069,7929.294659,4.736730
min,5711.300641,-41125.700000,16377.700000,1946.800730,0.000000
25%,23570.884288,400.000000,39514.525000,6412.817148,0.000000
50%,34696.543359,600.000000,65453.900000,11175.541084,0.000000
75%,47818.685031,900.000000,97032.150000,17155.004176,0.000000
max,81192.758824,24483.300000,209637.000000,37690.411210,124.000000


In [67]:
(balance_features['n_negative_balance_transactions'] > 0).value_counts()

n_negative_balance_transactions
False    4212
True      288
Name: count, dtype: int64

In [68]:
balance_features['has_negative_balance'] = (
    balance_features['n_negative_balance_transactions'] > 0
)

balance_features = balance_features.merge(
    transaction_activity[
        ['account_id', 'n_transactions']
    ],
    on='account_id',
    how='left',
    validate='one_to_one'
)

balance_features['negative_balance_ratio'] = (
    balance_features['n_negative_balance_transactions']
    / balance_features['n_transactions']
)

In [69]:
balance_features[
    [
        'has_negative_balance',
        'negative_balance_ratio'
    ]
].describe(include='all')

,has_negative_balance,negative_balance_ratio
count,4500,4500.000000
unique,2,NaN
top,False,NaN
freq,4212,NaN
mean,NaN,0.002885
std,NaN,0.019018
min,NaN,0.000000
25%,NaN,0.000000
50%,NaN,0.000000
75%,NaN,0.000000


### 1.7.4 Operation Mix

Operation-mix features describe how each account is used. The proportions are calculated over all transactions associated with the account. Transactions without a recorded operation remain part of the denominator, so they are not incorrectly assigned to another operation category.

In [70]:
query = '''
SELECT
    account_id,

    AVG(
        CASE WHEN operation = 'CASH_DEPOSIT' THEN 1.0 ELSE 0.0 END
    ) AS cash_deposit_ratio,

    AVG(
        CASE WHEN operation = 'CASH_WITHDRAWAL' THEN 1.0 ELSE 0.0 END
    ) AS cash_withdrawal_ratio,

    AVG(
        CASE WHEN operation = 'CARD_WITHDRAWAL' THEN 1.0 ELSE 0.0 END
    ) AS card_withdrawal_ratio,

    AVG(
        CASE WHEN operation = 'TRANSFER_IN' THEN 1.0 ELSE 0.0 END
    ) AS transfer_in_ratio,

    AVG(
        CASE WHEN operation = 'TRANSFER_OUT' THEN 1.0 ELSE 0.0 END
    ) AS transfer_out_ratio

FROM bank_transaction
GROUP BY account_id
ORDER BY account_id;
'''

operation_features = pd.read_sql(
    query,
    engine
)

operation_features.head()

,account_id,cash_deposit_ratio,cash_withdrawal_ratio,card_withdrawal_ratio,transfer_in_ratio,transfer_out_ratio
0,1,0.050209,0.405858,0.0,0.188285,0.167364
1,2,0.025105,0.495816,0.0,0.146444,0.186192
2,3,0.239316,0.273504,0.0,0.000000,0.333333
3,4,0.005376,0.333333,0.0,0.182796,0.317204
4,5,0.011905,0.416667,0.0,0.226190,0.166667


In [71]:
operation_features[
    [
        'cash_deposit_ratio',
        'cash_withdrawal_ratio',
        'card_withdrawal_ratio',
        'transfer_in_ratio',
        'transfer_out_ratio'
    ]
].describe()

,cash_deposit_ratio,cash_withdrawal_ratio,card_withdrawal_ratio,transfer_in_ratio,transfer_out_ratio
count,4500.000000,4500.000000,4500.000000,4500.000000,4500.000000
mean,0.155706,0.410119,0.007081,0.068500,0.178134
std,0.121769,0.083842,0.021113,0.094068,0.121611
min,0.001757,0.076923,0.000000,0.000000,0.000000
25%,0.013333,0.352764,0.000000,0.000000,0.117708
50%,0.181412,0.403479,0.000000,0.000000,0.172027
75%,0.233533,0.462687,0.000000,0.176471,0.268684
max,0.923077,0.682018,0.221239,0.300000,0.503937


### 1.8 Consolidated Transaction Features

The four transaction blocks are combined at `account_id` level. Redundant intermediate variables are excluded from the final feature table: monthly flows are retained instead of total flows, and the negative-balance ratio is retained instead of the underlying negative-balance transaction count.

In [72]:
transaction_features = (
    transaction_activity[
        [
            'account_id',
            'n_transactions',
            'first_transaction_date',
            'last_transaction_date',
            'active_days',
            'observation_days',
            'transactions_per_month',
            'active_day_ratio'
        ]
    ]
    .merge(
        transaction_flows[
            [
                'account_id',
                'n_inflows',
                'monthly_inflow',
                'monthly_outflow',
                'avg_inflow_amount',
                'avg_outflow_amount'
            ]
        ],
        on='account_id',
        validate='one_to_one'
    )
    .merge(
        balance_features[
            [
                'account_id',
                'avg_balance',
                'min_balance',
                'max_balance',
                'balance_std',
                'has_negative_balance',
                'negative_balance_ratio'
            ]
        ],
        on='account_id',
        validate='one_to_one'
    )
    .merge(
        operation_features,
        on='account_id',
        validate='one_to_one'
    )
)

transaction_features.head()

,account_id,n_transactions,first_transaction_date,last_transaction_date,active_days,observation_days,transactions_per_month,active_day_ratio,n_inflows,monthly_inflow,...,min_balance,max_balance,balance_std,has_negative_balance,negative_balance_ratio,cash_deposit_ratio,cash_withdrawal_ratio,card_withdrawal_ratio,transfer_in_ratio,transfer_out_ratio
0,1,239,1995-03-24,1998-12-31,187,1379,5.275678,0.135606,102,4289.424235,...,1000.0,30414.8,3656.890437,False,0.0,0.050209,0.405858,0.0,0.188285,0.167364
1,2,478,1993-02-26,1998-12-31,369,2135,6.815138,0.172834,152,22770.167934,...,1100.0,69302.0,11028.141344,False,0.0,0.025105,0.495816,0.0,0.146444,0.186192
2,3,117,1997-07-07,1998-12-31,100,543,6.558895,0.184162,46,9701.502689,...,1000.0,53446.5,10508.662466,False,0.0,0.239316,0.273504,0.0,0.000000,0.333333
3,4,186,1996-02-21,1998-12-31,151,1045,5.418029,0.144498,65,5602.882526,...,800.0,34870.0,5009.975052,False,0.0,0.005376,0.333333,0.0,0.182796,0.317204
4,5,84,1997-05-30,1998-12-31,69,581,4.400964,0.118761,35,5107.475731,...,600.0,32036.2,4752.471004,False,0.0,0.011905,0.416667,0.0,0.226190,0.166667


In [73]:
transaction_features.shape

(4500, 24)

### 1.9 Customer 360 Dataset

The final Customer Intelligence dataset combines account, customer, product, socioeconomic, and transaction information. Product attributes remain missing when the corresponding product is not held; these are structural missing values rather than data-quality errors.

In [74]:
customer_360 = customer_base.drop(
    columns='has_standing_order'
).merge(
    transaction_features,
    on='account_id',
    how='left',
    validate='one_to_one'
)

assert (
    len(customer_360)
    == customer_360['account_id'].nunique()
    == 4500
)

In [75]:
customer_360.head()

,account_id,account_open_date,frequency,owner_client_id,birth_date,gender,age,account_district_id,client_district_id,district_mismatch,...,min_balance,max_balance,balance_std,has_negative_balance,negative_balance_ratio,cash_deposit_ratio,cash_withdrawal_ratio,card_withdrawal_ratio,transfer_in_ratio,transfer_out_ratio
0,1,1995-03-24,MONTHLY,1,1970-12-13,Female,28,18,18,False,...,1000.0,30414.8,3656.890437,False,0.0,0.050209,0.405858,0.0,0.188285,0.167364
1,2,1993-02-26,MONTHLY,2,1945-02-04,Male,53,1,1,False,...,1100.0,69302.0,11028.141344,False,0.0,0.025105,0.495816,0.0,0.146444,0.186192
2,3,1997-07-07,MONTHLY,4,1956-12-01,Male,42,5,5,False,...,1000.0,53446.5,10508.662466,False,0.0,0.239316,0.273504,0.0,0.000000,0.333333
3,4,1996-02-21,MONTHLY,6,1919-09-22,Male,79,12,12,False,...,800.0,34870.0,5009.975052,False,0.0,0.005376,0.333333,0.0,0.182796,0.317204
4,5,1997-05-30,MONTHLY,7,1929-01-25,Male,69,15,15,False,...,600.0,32036.2,4752.471004,False,0.0,0.011905,0.416667,0.0,0.226190,0.166667


In [76]:
customer_360.shape

(4500, 55)

In [77]:
customer_360.columns.tolist()

['account_id',
 'account_open_date',
 'frequency',
 'owner_client_id',
 'birth_date',
 'gender',
 'age',
 'account_district_id',
 'client_district_id',
 'district_mismatch',
 'district_name',
 'region',
 'inhabitants',
 'urban_ratio',
 'average_salary',
 'unemployment_rate_96',
 'entrepreneurs_per_1000',
 'crimes_96',
 'has_disponent',
 'card_type',
 'has_card',
 'n_standing_orders',
 'total_standing_order_amount',
 'household_payment_amount',
 'insurance_payment_amount',
 'leasing_payment_amount',
 'loan_payment_amount',
 'uncategorized_order_amount',
 'loan_amount',
 'loan_duration',
 'loan_payment',
 'has_loan',
 'n_transactions',
 'first_transaction_date',
 'last_transaction_date',
 'active_days',
 'observation_days',
 'transactions_per_month',
 'active_day_ratio',
 'n_inflows',
 'monthly_inflow',
 'monthly_outflow',
 'avg_inflow_amount',
 'avg_outflow_amount',
 'avg_balance',
 'min_balance',
 'max_balance',
 'balance_std',
 'has_negative_balance',
 'negative_balance_ratio',
 'cash

In [78]:
customer_360.isna().sum()[
    customer_360.isna().sum() > 0
]

card_type        3608
loan_amount      3818
loan_duration    3818
loan_payment     3818
dtype: int64

## 2. Credit Risk Analytical Dataset

### 2.1 Loan Population and Target

The analytical dataset retains all 682 loans, with one observation per `loan_id`. `target_default` represents a definitive supervised outcome only for completed loans:

- `A`: finished without repayment problems — target 0.
- `B`: finished but not repaid — target 1.
- `C`: active without repayment problems — target `pd.NA`.
- `D`: active with debt — target `pd.NA`.

C and D are unfinished/active loans in the extract. Both remain in the analytical dataset, but neither is treated as a definitive supervised outcome. Missing targets are intentional and must not be imputed as 0 or 1. `target_default` uses nullable `Int64`; `loan_status` is retained for validation and must not be used as a predictor.

The later Credit Risk Modeling notebook must construct its modeling sample using `loan_status IN ('A', 'B')`, equivalently `credit_risk['target_default'].notna()`, with A=0 and B=1. It must preserve the existing temporal restrictions: features use only information strictly before `loan_date`. This notebook does not filter loans or perform modeling, resampling, train/test splitting, or feature selection.


In [79]:
query = '''
SELECT
    l.loan_id,
    l.account_id,
    l.date AS loan_date,
    l.amount AS loan_amount,
    l.duration AS loan_duration,
    l.payments AS loan_payment,
    l.status AS loan_status,

    a.date AS account_open_date,
    c.birth_date,
    c.gender

FROM loan l

INNER JOIN account a
    ON l.account_id = a.account_id

INNER JOIN disp d
    ON a.account_id = d.account_id
    AND d.type = 'OWNER'

INNER JOIN client c
    ON d.client_id = c.client_id

ORDER BY l.loan_id;
'''

credit_risk_base = pd.read_sql(
    query,
    engine,
    parse_dates=[
        'loan_date',
        'account_open_date',
        'birth_date'
    ]
)

credit_risk_base.head()

,loan_id,account_id,loan_date,loan_amount,loan_duration,loan_payment,loan_status,account_open_date,birth_date,gender
0,4959,2,1994-01-05,80952.0,24,3373.0,A,1993-02-26,1945-02-04,Male
1,4961,19,1996-04-29,30276.0,12,2523.0,B,1995-04-07,1939-04-23,Female
2,4962,25,1997-12-08,30276.0,12,2523.0,A,1996-07-28,1962-02-09,Male
3,4967,37,1998-10-14,318480.0,60,5308.0,D,1997-08-18,1952-08-26,Male
4,4968,38,1998-04-19,110736.0,48,2307.0,C,1997-08-08,1940-01-30,Female


In [80]:
assert len(credit_risk_base) == credit_risk_base['loan_id'].nunique() == 682
assert credit_risk_base['loan_status'].isin(['A', 'B', 'C', 'D']).all()

loan_status_counts = (
    credit_risk_base['loan_status']
    .value_counts().reindex(['A', 'B', 'C', 'D'], fill_value=0)
)
loan_status_counts


loan_status
A    203
B     31
C    403
D     45
Name: count, dtype: int64

In [81]:
credit_risk_base['target_default'] = (
    credit_risk_base['loan_status']
    .map({'A': 0, 'B': 1})
    .astype('Int64')
)

credit_risk_base[
    ['loan_status', 'target_default']
].value_counts(dropna=False).sort_index()


loan_status  target_default
A            0                 203
B            1                  31
C            <NA>              403
D            <NA>               45
Name: count, dtype: int64

### 2.2 Origination-Time Features

Age and account tenure are calculated at the loan origination date. The current customer age stored in the database is not used because it refers to the end of the data extract.

In [82]:
credit_risk_base['age_at_loan'] = (
    credit_risk_base['loan_date'].dt.year
    - credit_risk_base['birth_date'].dt.year
    - (
        credit_risk_base['loan_date'].dt.strftime('%m%d')
        < credit_risk_base['birth_date'].dt.strftime('%m%d')
    ).astype(int)
)

credit_risk_base['account_tenure_days'] = (
    credit_risk_base['loan_date']
    - credit_risk_base['account_open_date']
).dt.days

credit_risk_base[
    ['age_at_loan', 'account_tenure_days']
].describe()

,age_at_loan,account_tenure_days
count,682.000000,682.000000
mean,37.630499,398.240469
std,12.768289,164.611359
min,13.000000,102.000000
25%,27.000000,261.250000
50%,37.000000,395.500000
75%,48.000000,528.750000
max,61.000000,697.000000


### 2.3 Pre-Loan Transaction Features

Only transactions strictly earlier than `loan_date` are used. Transactions occurring on the origination date are excluded because their ordering relative to loan approval is unknown. Recent activity covers the 90 days immediately preceding origination.

In [83]:
query = '''
SELECT
    l.loan_id,

    COUNT(*) AS preloan_n_transactions,
    MIN(t.date) AS preloan_first_transaction_date,
    MAX(t.date) AS preloan_last_transaction_date,
    COUNT(DISTINCT t.date) AS preloan_active_days,

    COUNT(*) FILTER (
        WHERE t.type = 'CREDIT'
    ) AS preloan_n_inflows,

    COUNT(*) FILTER (
        WHERE t.type IN ('DEBIT', 'WITHDRAWAL')
    ) AS preloan_n_outflows,

    SUM(t.amount) FILTER (
        WHERE t.type = 'CREDIT'
    ) AS preloan_total_inflow,

    SUM(t.amount) FILTER (
        WHERE t.type IN ('DEBIT', 'WITHDRAWAL')
    ) AS preloan_total_outflow,

    AVG(t.balance) AS preloan_avg_balance,
    MIN(t.balance) AS preloan_min_balance,
    MAX(t.balance) AS preloan_max_balance,
    STDDEV_SAMP(t.balance) AS preloan_balance_std,

    AVG(
        CASE WHEN t.balance < 0 THEN 1.0 ELSE 0.0 END
    ) AS preloan_negative_balance_ratio,

    AVG(
        CASE WHEN t.operation = 'CASH_DEPOSIT' THEN 1.0 ELSE 0.0 END
    ) AS preloan_cash_deposit_ratio,

    AVG(
        CASE WHEN t.operation = 'CASH_WITHDRAWAL' THEN 1.0 ELSE 0.0 END
    ) AS preloan_cash_withdrawal_ratio,

    AVG(
        CASE WHEN t.operation = 'CARD_WITHDRAWAL' THEN 1.0 ELSE 0.0 END
    ) AS preloan_card_withdrawal_ratio,

    AVG(
        CASE WHEN t.operation = 'TRANSFER_IN' THEN 1.0 ELSE 0.0 END
    ) AS preloan_transfer_in_ratio,

    AVG(
        CASE WHEN t.operation = 'TRANSFER_OUT' THEN 1.0 ELSE 0.0 END
    ) AS preloan_transfer_out_ratio,

    COUNT(*) FILTER (
        WHERE t.date >= l.date - INTERVAL '90 days'
    ) AS transactions_90d,

    SUM(t.amount) FILTER (
        WHERE t.type = 'CREDIT'
        AND t.date >= l.date - INTERVAL '90 days'
    ) AS inflow_90d,

    SUM(t.amount) FILTER (
        WHERE t.type IN ('DEBIT', 'WITHDRAWAL')
        AND t.date >= l.date - INTERVAL '90 days'
    ) AS outflow_90d

FROM loan l

INNER JOIN bank_transaction t
    ON l.account_id = t.account_id
    AND t.date < l.date

GROUP BY l.loan_id
ORDER BY l.loan_id;
'''

preloan_transaction_features = pd.read_sql(
    query,
    engine,
    parse_dates=[
        'preloan_first_transaction_date',
        'preloan_last_transaction_date'
    ]
)

preloan_transaction_features.head()

,loan_id,preloan_n_transactions,preloan_first_transaction_date,preloan_last_transaction_date,preloan_active_days,preloan_n_inflows,preloan_n_outflows,preloan_total_inflow,preloan_total_outflow,preloan_avg_balance,...,preloan_balance_std,preloan_negative_balance_ratio,preloan_cash_deposit_ratio,preloan_cash_withdrawal_ratio,preloan_card_withdrawal_ratio,preloan_transfer_in_ratio,preloan_transfer_out_ratio,transactions_90d,inflow_90d,outflow_90d
0,4959,54,1993-02-26,1993-12-31,48,22,32,228685.3,200844.6,32590.624074,...,12061.705682,0.000000,0.037037,0.500000,0.0,0.185185,0.092593,19,71204.4,53871.8
1,4961,80,1995-04-07,1996-04-10,54,46,34,242179.2,226324.8,25197.092500,...,15039.248405,0.000000,0.112500,0.425000,0.0,0.162500,0.000000,21,53573.5,55543.8
2,4962,164,1996-07-28,1997-12-06,128,51,113,1009153.3,977531.4,62991.264634,...,22476.347980,0.000000,0.219512,0.426829,0.0,0.000000,0.262195,28,161258.6,183430.8
3,4967,116,1997-08-18,1998-10-12,97,32,84,447359.9,430992.0,39953.970690,...,23451.647180,0.008621,0.155172,0.413793,0.0,0.000000,0.310345,27,90958.1,89842.2
4,4968,55,1997-08-08,1998-04-10,43,23,32,148403.7,110987.0,31383.449091,...,10950.811533,0.000000,0.109091,0.345455,0.0,0.163636,0.236364,26,49770.6,37866.0


In [84]:
preloan_transaction_features['preloan_observation_days'] = (
    preloan_transaction_features['preloan_last_transaction_date']
    - preloan_transaction_features['preloan_first_transaction_date']
).dt.days + 1

preloan_observation_months = (
    preloan_transaction_features['preloan_observation_days']
    / 30.44
)

preloan_transaction_features['preloan_transactions_per_month'] = (
    preloan_transaction_features['preloan_n_transactions']
    / preloan_observation_months
)

preloan_transaction_features['preloan_monthly_inflow'] = (
    preloan_transaction_features['preloan_total_inflow']
    / preloan_observation_months
)

preloan_transaction_features['preloan_monthly_outflow'] = (
    preloan_transaction_features['preloan_total_outflow']
    / preloan_observation_months
)

preloan_transaction_features['preloan_avg_inflow_amount'] = (
    preloan_transaction_features['preloan_total_inflow']
    / preloan_transaction_features['preloan_n_inflows']
)

preloan_transaction_features['preloan_avg_outflow_amount'] = (
    preloan_transaction_features['preloan_total_outflow']
    / preloan_transaction_features['preloan_n_outflows']
)

In [85]:
preloan_transaction_features[
    [
        'preloan_n_transactions',
        'preloan_transactions_per_month',
        'preloan_monthly_inflow',
        'preloan_monthly_outflow',
        'preloan_avg_balance',
        'preloan_negative_balance_ratio',
        'transactions_90d'
    ]
].describe()

,preloan_n_transactions,preloan_transactions_per_month,preloan_monthly_inflow,preloan_monthly_outflow,preloan_avg_balance,preloan_negative_balance_ratio,transactions_90d
count,682.000000,682.000000,682.000000,679.000000,682.000000,682.000000,682.000000
mean,80.196481,5.878393,29717.758582,25601.274775,43606.789462,0.002039,20.747801
std,44.777664,1.654022,15226.477998,14613.175958,13481.855233,0.014229,6.198458
min,2.000000,0.669011,3378.505495,996.218182,5350.000000,0.000000,1.000000
25%,45.000000,4.940865,17645.184451,14432.033996,32833.034787,0.000000,17.000000
50%,77.000000,5.775456,26008.456528,21800.951170,43159.249595,0.000000,20.500000
75%,112.000000,6.926392,40175.818648,35804.723557,53613.958260,0.000000,25.000000
max,218.000000,11.543837,83237.384551,77838.712757,79500.538095,0.218750,40.000000


### 2.4 Credit Risk Dataset

Loan, borrower, and pre-origination transaction features are combined at `loan_id` level. The original loan status is retained to document the target, but it must not be used as a predictor.

In [86]:
credit_risk = credit_risk_base.merge(
    preloan_transaction_features,
    on='loan_id',
    how='left',
    validate='one_to_one'
)

assert (
    len(credit_risk)
    == credit_risk['loan_id'].nunique()
    == 682
)
assert (
    credit_risk['preloan_last_transaction_date']
    < credit_risk['loan_date']
).all()

In [87]:
credit_risk.head()

,loan_id,account_id,loan_date,loan_amount,loan_duration,loan_payment,loan_status,account_open_date,birth_date,gender,...,preloan_transfer_out_ratio,transactions_90d,inflow_90d,outflow_90d,preloan_observation_days,preloan_transactions_per_month,preloan_monthly_inflow,preloan_monthly_outflow,preloan_avg_inflow_amount,preloan_avg_outflow_amount
0,4959,2,1994-01-05,80952.0,24,3373.0,A,1993-02-26,1945-02-04,Male,...,0.092593,19,71204.4,53871.8,309,5.319612,22528.092337,19785.468039,10394.786364,6276.393750
1,4961,19,1996-04-29,30276.0,12,2523.0,B,1995-04-07,1939-04-23,Female,...,0.000000,21,53573.5,55543.8,370,6.581622,19924.148238,18619.802465,5264.765217,6656.611765
2,4962,25,1997-12-08,30276.0,12,2523.0,A,1996-07-28,1962-02-09,Male,...,0.262195,28,161258.6,183430.8,497,10.044588,61808.101513,59871.339670,19787.319608,8650.720354
3,4967,37,1998-10-14,318480.0,60,5308.0,D,1997-08-18,1952-08-26,Male,...,0.310345,27,90958.1,89842.2,421,8.387268,32345.927211,31162.461948,13979.996875,5130.857143
4,4968,38,1998-04-19,110736.0,48,2307.0,C,1997-08-08,1940-01-30,Female,...,0.236364,26,49770.6,37866.0,246,6.805691,18363.449707,13733.513333,6452.334783,3468.343750


In [88]:
credit_risk.shape

(682, 40)

In [89]:
credit_risk.columns.tolist()

['loan_id',
 'account_id',
 'loan_date',
 'loan_amount',
 'loan_duration',
 'loan_payment',
 'loan_status',
 'account_open_date',
 'birth_date',
 'gender',
 'target_default',
 'age_at_loan',
 'account_tenure_days',
 'preloan_n_transactions',
 'preloan_first_transaction_date',
 'preloan_last_transaction_date',
 'preloan_active_days',
 'preloan_n_inflows',
 'preloan_n_outflows',
 'preloan_total_inflow',
 'preloan_total_outflow',
 'preloan_avg_balance',
 'preloan_min_balance',
 'preloan_max_balance',
 'preloan_balance_std',
 'preloan_negative_balance_ratio',
 'preloan_cash_deposit_ratio',
 'preloan_cash_withdrawal_ratio',
 'preloan_card_withdrawal_ratio',
 'preloan_transfer_in_ratio',
 'preloan_transfer_out_ratio',
 'transactions_90d',
 'inflow_90d',
 'outflow_90d',
 'preloan_observation_days',
 'preloan_transactions_per_month',
 'preloan_monthly_inflow',
 'preloan_monthly_outflow',
 'preloan_avg_inflow_amount',
 'preloan_avg_outflow_amount']

#### Structural Missing Values in Pre-Loan Outflows

The three confirmed loan/account pairs (5063/442, 5314/1787, and 6594/7824) have `preloan_n_outflows = 0`: no outflows were observed before `loan_date`. These are structural missing values, so `preloan_total_outflow`, `outflow_90d`, and `preloan_monthly_outflow` are filled with 0. `preloan_avg_outflow_amount` remains NaN because the average is undefined when `n_outflows = 0`.


In [90]:
zero_outflow_columns = [
    'preloan_total_outflow',
    'outflow_90d',
    'preloan_monthly_outflow'
]

credit_risk[zero_outflow_columns] = (
    credit_risk[zero_outflow_columns]
    .fillna(0)
)

credit_risk.isna().sum()[
    credit_risk.isna().sum() > 0
]

target_default                448
preloan_avg_outflow_amount      3
dtype: int64

In [91]:
credit_risk.loc[
    credit_risk['preloan_total_outflow'].isna(),
    [
        'loan_id',
        'account_id',
        'loan_date',
        'preloan_n_transactions',
        'preloan_n_inflows',
        'preloan_n_outflows',
        'preloan_total_inflow',
        'preloan_total_outflow',
        'outflow_90d'
    ]
]

,loan_id,account_id,loan_date,preloan_n_transactions,preloan_n_inflows,preloan_n_outflows,preloan_total_inflow,preloan_total_outflow,outflow_90d


In [92]:
credit_risk.loc[
    credit_risk['age_at_loan'] < 18,
    [
        'loan_id',
        'account_id',
        'birth_date',
        'loan_date',
        'age_at_loan',
        'loan_amount',
        'loan_duration',
        'loan_status'
    ]
]

,loan_id,account_id,birth_date,loan_date,age_at_loan,loan_amount,loan_duration,loan_status
120,5385,2143,1980-08-31,1996-09-09,16,71460.0,36,C
136,5445,2322,1979-02-14,1996-06-29,17,46320.0,24,A
139,5449,2343,1980-07-09,1995-11-04,15,99336.0,24,A
177,5593,3037,1980-09-17,1996-04-18,15,277884.0,36,D
199,5681,3448,1980-02-19,1997-10-20,17,68664.0,24,C
206,5713,3629,1978-01-31,1995-05-30,17,158760.0,60,C
267,5938,4650,1980-08-06,1996-08-25,16,402000.0,60,C
277,5977,4803,1979-12-10,1997-02-25,17,59136.0,48,C
282,5992,4863,1979-04-06,1997-03-18,17,76080.0,12,A
286,6006,4948,1979-05-31,1996-11-15,17,359760.0,60,C


In [93]:
underage_loans = (
    credit_risk['age_at_loan'] < 18
).sum()

underage_loans

np.int64(30)

#### Age-at-Loan Validation

The derived `age_at_loan` variable includes borrowers younger than 18 years at the recorded loan date.

The source `birth_number` encoding and the date-of-birth transformation were verified against the PKDD'99 Financial Dataset convention. The observed ages therefore reflect the source data rather than an identified transformation error.

These observations are retained without modification. Their potential treatment as anomalous observations should be evaluated during the modeling stage rather than corrected at the analytical-dataset construction stage.

### 2.5 Temporal Cutoff Validation

The following check summarizes all transactions associated with loan accounts. Only the transactions in `transactions_before_loan` were used to construct the Credit Risk features.

In [94]:
query = '''
SELECT
    COUNT(*) FILTER (
        WHERE t.date < l.date
    ) AS transactions_before_loan,

    COUNT(*) FILTER (
        WHERE t.date = l.date
    ) AS transactions_on_loan_date,

    COUNT(*) FILTER (
        WHERE t.date > l.date
    ) AS transactions_after_loan

FROM loan l

INNER JOIN bank_transaction t
    ON l.account_id = t.account_id;
'''

cutoff_validation = pd.read_sql(
    query,
    engine
)

cutoff_validation

,transactions_before_loan,transactions_on_loan_date,transactions_after_loan
0,54694,166,136696


### 2.6 Final Results

In [95]:
customer_360

,account_id,account_open_date,frequency,owner_client_id,birth_date,gender,age,account_district_id,client_district_id,district_mismatch,...,min_balance,max_balance,balance_std,has_negative_balance,negative_balance_ratio,cash_deposit_ratio,cash_withdrawal_ratio,card_withdrawal_ratio,transfer_in_ratio,transfer_out_ratio
0,1,1995-03-24,MONTHLY,1,1970-12-13,Female,28,18,18,False,...,1000.0,30414.8,3656.890437,False,0.000000,0.050209,0.405858,0.000000,0.188285,0.167364
1,2,1993-02-26,MONTHLY,2,1945-02-04,Male,53,1,1,False,...,1100.0,69302.0,11028.141344,False,0.000000,0.025105,0.495816,0.000000,0.146444,0.186192
2,3,1997-07-07,MONTHLY,4,1956-12-01,Male,42,5,5,False,...,1000.0,53446.5,10508.662466,False,0.000000,0.239316,0.273504,0.000000,0.000000,0.333333
3,4,1996-02-21,MONTHLY,6,1919-09-22,Male,79,12,12,False,...,800.0,34870.0,5009.975052,False,0.000000,0.005376,0.333333,0.000000,0.182796,0.317204
4,5,1997-05-30,MONTHLY,7,1929-01-25,Male,69,15,15,False,...,600.0,32036.2,4752.471004,False,0.000000,0.011905,0.416667,0.000000,0.226190,0.166667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4495,11333,1994-05-26,MONTHLY,13931,1942-01-01,Male,56,8,8,False,...,-1291.0,137043.2,29645.126065,True,0.029891,0.211957,0.502717,0.000000,0.000000,0.000000
4496,11349,1995-05-26,WEEKLY,13955,1945-10-30,Female,53,1,1,False,...,200.0,113678.7,20573.358546,False,0.000000,0.016447,0.447368,0.000000,0.141447,0.253289
4497,11359,1994-10-01,MONTHLY,13968,1968-04-13,Male,30,61,61,False,...,1000.0,81705.8,16732.929108,False,0.000000,0.005291,0.391534,0.058201,0.134921,0.275132
4498,11362,1995-10-14,MONTHLY,13971,1962-10-19,Female,36,67,67,False,...,1000.0,56055.6,10092.374189,False,0.000000,0.125000,0.299419,0.000000,0.000000,0.465116


In [96]:
credit_risk

,loan_id,account_id,loan_date,loan_amount,loan_duration,loan_payment,loan_status,account_open_date,birth_date,gender,...,preloan_transfer_out_ratio,transactions_90d,inflow_90d,outflow_90d,preloan_observation_days,preloan_transactions_per_month,preloan_monthly_inflow,preloan_monthly_outflow,preloan_avg_inflow_amount,preloan_avg_outflow_amount
0,4959,2,1994-01-05,80952.0,24,3373.0,A,1993-02-26,1945-02-04,Male,...,0.092593,19,71204.4,53871.8,309,5.319612,22528.092337,19785.468039,10394.786364,6276.393750
1,4961,19,1996-04-29,30276.0,12,2523.0,B,1995-04-07,1939-04-23,Female,...,0.000000,21,53573.5,55543.8,370,6.581622,19924.148238,18619.802465,5264.765217,6656.611765
2,4962,25,1997-12-08,30276.0,12,2523.0,A,1996-07-28,1962-02-09,Male,...,0.262195,28,161258.6,183430.8,497,10.044588,61808.101513,59871.339670,19787.319608,8650.720354
3,4967,37,1998-10-14,318480.0,60,5308.0,D,1997-08-18,1952-08-26,Male,...,0.310345,27,90958.1,89842.2,421,8.387268,32345.927211,31162.461948,13979.996875,5130.857143
4,4968,38,1998-04-19,110736.0,48,2307.0,C,1997-08-08,1940-01-30,Female,...,0.236364,26,49770.6,37866.0,246,6.805691,18363.449707,13733.513333,6452.334783,3468.343750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
677,7294,11327,1998-09-27,39168.0,24,1632.0,C,1997-10-15,1979-02-22,Female,...,0.111111,20,61708.2,54622.8,334,4.921437,22462.250168,16800.838515,10715.865217,5946.632258
678,7295,11328,1998-07-18,280440.0,60,4674.0,C,1996-11-05,1952-09-09,Female,...,0.000000,19,61359.6,65226.8,612,5.272288,22233.251654,20676.270523,9510.687234,7045.745763
679,7304,11349,1995-10-29,419880.0,60,6998.0,C,1995-05-26,1945-10-30,Female,...,0.055556,12,126981.5,161186.0,151,3.628609,46807.144503,41826.979073,21108.227273,29640.857143
680,7305,11359,1996-08-06,54024.0,12,4502.0,A,1994-10-01,1968-04-13,Male,...,0.231293,22,94967.7,98504.8,670,6.678627,29664.570531,28497.737182,14509.608889,6149.507843


### 2.7 Supervised Target Validation

All 682 loans remain in `credit_risk`. The cross-tab explicitly includes missing definitive targets for C/D. The summary separates completed loans eligible for the future supervised cohort from active loans retained only in the full analytical dataset.


In [97]:
assert len(credit_risk) == credit_risk['loan_id'].nunique() == 682
assert credit_risk['loan_status'].isin(['A', 'B', 'C', 'D']).all()
assert credit_risk['target_default'].notna().equals(
    credit_risk['loan_status'].isin(['A', 'B'])
)
assert credit_risk.loc[credit_risk['loan_status'].eq('A'), 'target_default'].eq(0).all()
assert credit_risk.loc[credit_risk['loan_status'].eq('B'), 'target_default'].eq(1).all()
assert credit_risk.loc[credit_risk['loan_status'].isin(['C', 'D']), 'target_default'].isna().all()

loan_status_target_validation = pd.crosstab(
    credit_risk['loan_status'],
    credit_risk['target_default'].astype('string').fillna('pd.NA'),
).reindex(index=['A', 'B', 'C', 'D'], columns=['0', '1', 'pd.NA'], fill_value=0)
loan_status_target_validation


target_default,0,1,pd.NA
loan_status,,,
A,203,0,0
B,0,31,0
C,0,0,403
D,0,0,45


In [98]:
target_availability = pd.Series({
    'Known target (A/B)': int(credit_risk['target_default'].notna().sum()),
    'No definitive target (C/D)': int(credit_risk['target_default'].isna().sum()),
    'All analytical loans': len(credit_risk),
}, name='n_loans')
assert target_availability.iloc[:2].sum() == 682
assert (credit_risk['preloan_last_transaction_date'] < credit_risk['loan_date']).all()
target_availability


Known target (A/B)            234
No definitive target (C/D)    448
All analytical loans          682
Name: n_loans, dtype: int64

In [99]:
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

customer_360.to_parquet(
    PROCESSED_DIR / 'customer_360.parquet',
    index=False
)

credit_risk.to_parquet(
    PROCESSED_DIR / 'credit_risk.parquet',
    index=False
)

print('Analytical datasets saved successfully.')

Analytical datasets saved successfully.
